# ICT Backtest + P39 Volume Analysis — Google Colab runner

Runs the **full 2022–2025 backtest** and the **P39 tick-volume analysis** directly on the data
already in your Google Drive. **No token, nothing to paste.**

### Before you run (30 seconds): make the repo public
So Colab can fetch the code without a password:
1. On your phone open **github.com/ThabisoCollinSengane/Ict** → **Settings**
2. Scroll to the bottom (**Danger Zone**) → **Change repository visibility** → **Public** → confirm.

That's the only setup. Then **Runtime ▸ Run all**. You'll tap **Allow** once for Drive; no token box.
**Switch the repo back to Private after it finishes.**

_Results are saved to a new `ICT_results` folder in your Drive and also printed at the end so you_
_can copy them straight to Claude._


### 1. Mount your Google Drive  (tap Allow on the popup)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


### 2. Fetch the code (no token — needs the repo set to Public, see top)


In [ ]:
import os, glob, subprocess
REPO   = 'ThabisoCollinSengane/Ict'
BRANCH = 'p39-volume-analysis'
if os.path.isdir('/content/Ict'):
    subprocess.run(['rm','-rf','/content/Ict'])
r = subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',
                    f'https://github.com/{REPO}.git','/content/Ict'],
                   capture_output=True, text=True)
print(r.stderr[-600:] or 'cloned')
assert os.path.isdir('/content/Ict/scripts'), \
    'Clone failed — is the repo set to Public? (Settings > Danger Zone > visibility)'
os.chdir('/content/Ict')
print('Repo ready at', os.getcwd())


### 3. Install the Python dependencies (~1 min)


In [ ]:
import subprocess
p = subprocess.run(['pip','install','-q','-r','requirements.txt'], capture_output=True, text=True)
print(p.stdout[-300:]); print(p.stderr[-300:]); print('dependencies installed')


### 4. Locate your data folders in Drive
Auto-finds the M1 folder and the (separate) tick folder. If either shows 0 zips, edit the path and re-run.


In [ ]:
import glob, os
def find_dir(tokens, must_glob):
    for d in glob.glob('/content/drive/MyDrive/*/'):
        b = os.path.basename(d.rstrip('/')).lower()
        if all(t in b for t in tokens) and glob.glob(os.path.join(d, must_glob)):
            return d.rstrip('/')
    for root, _d, _f in os.walk('/content/drive/MyDrive'):
        b = os.path.basename(root).lower()
        if all(t in b for t in tokens) and glob.glob(os.path.join(root, must_glob)):
            return root
    return None
M1_DIR   = find_dir(['backtesting','data'], 'HISTDATA_*_M1*.zip')
TICK_DIR = find_dir(['tick','data'],        'HISTDATA_*_T*.zip')
print('M1 folder  :', M1_DIR, '->', len(glob.glob(f'{M1_DIR}/HISTDATA_*_M1*.zip')) if M1_DIR else 0, 'zips')
print('Tick folder:', TICK_DIR, '->', len(glob.glob(f'{TICK_DIR}/HISTDATA_*_T*.zip')) if TICK_DIR else 0, 'zips')
assert M1_DIR and TICK_DIR, 'One folder not found — set the path manually above and re-run.'


### 5. Prepare the M1 data
Unzips + converts (incl. UDXUSD MT→ASCII) + renames into `data/histdata/`. Look for `OK — runnable`.


In [ ]:
!python scripts/prepare_histdata.py "{M1_DIR}"


### 6. Run the full backtest (current algorithm, 2022–2025)
The R429M / PF 4.47 / MaxDD −12.95% run — and it makes the trade dump P39 needs. Summary is at the end.


In [ ]:
import subprocess
res = subprocess.run(['python','run_backtest_histdata.py','--years','2022','2023','2024','2025'],
                     capture_output=True, text=True)
open('/content/backtest_output.txt','w').write(res.stdout)
print(res.stdout[-9000:])
if res.returncode != 0:
    print('--- STDERR ---'); print(res.stderr[-3000:])


### 7. P39 — aggregate the tick data (slow step; several minutes)


In [ ]:
!python scripts/p39_volume_analysis.py aggregate "{TICK_DIR}"


### 8. P39 — analyse and write the report


In [ ]:
!python scripts/p39_volume_analysis.py analyse


### 9. Save results to Drive AND print them here


In [ ]:
import shutil, os
OUT = '/content/drive/MyDrive/ICT_results'
os.makedirs(OUT, exist_ok=True)
for f in ['data/p39_volume_report.md','data/histdata/trades_dump.csv','/content/backtest_output.txt']:
    if os.path.exists(f): shutil.copy(f, OUT); print('saved ->', os.path.basename(f))
print('\n' + '='*60 + '\n  P39 REPORT (copy this to Claude)\n' + '='*60)
print(open('data/p39_volume_report.md').read() if os.path.exists('data/p39_volume_report.md') else '(no report)')


---
**Done?** Say *"results are ready"* to Claude, or copy the report printed above straight into the chat.
Then switch the repo back to **Private** (Settings → Danger Zone).
